In [ ]:
!pip install mujoco
!pip install mujoco_mjx
!pip install brax
!pip install playground
!pip install playground warp-lang==1.12.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.9/356.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.7/138.7 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 131.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.4/

In [ ]:
import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union
import numpy as np
import mediapy as media
import matplotlib.pyplot as plt

from datetime import datetime
import functools
import os
from typing import Any, Dict, Sequence, Tuple, Union
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from brax.training.agents.sac import networks as sac_networks
from brax.training.agents.sac import train as sac
from etils import epath
from flax import struct
from flax.training import orbax_utils
from IPython.display import HTML, clear_output
import jax
from jax import numpy as jp
from matplotlib import pyplot as plt
import mediapy as media
from ml_collections import config_dict
import mujoco
from mujoco import mjx
import numpy as np
from orbax import checkpoint as ocp

from mujoco_playground import registry
from mujoco_playground.config import dm_control_suite_params
from mujoco_playground import wrapper
from mujoco_playground._src.dm_control_suite.swimmer import Swim

In [ ]:
def train_agent(env_name, agent='ppo', seed=0):
    if env_name == "SwimmerSwimmer6":
      env = Swim(n_links=3)
    else:
      env = registry.load(env_name)

    if agent == 'ppo':
        params = dm_control_suite_params.brax_ppo_config(env_name)
        train_fn = ppo.train
        network_factory = ppo_networks.make_ppo_networks
        training_params = dict(params)
    else:
        params = dm_control_suite_params.brax_sac_config(env_name)
        train_fn = sac.train
        network_factory = sac_networks.make_sac_networks
        training_params = dict(params)
        training_params.pop('num_resets_per_eval', None)

    if "network_factory" in params:
        del training_params["network_factory"]
        network_factory = functools.partial(network_factory, **params.network_factory)

    rewards = []
    def progress(num_steps, metrics):
        rewards.append((num_steps, metrics["eval/episode_reward"]))
        print(f"[{agent.upper()}|{env_name}] steps={num_steps}, reward={metrics['eval/episode_reward']:.2f}")

    t0 = datetime.now()
    inference_fn, trained_params, metrics = functools.partial(
        train_fn, **training_params,
        network_factory=network_factory,
        progress_fn=progress, seed=seed
    )(environment=env, wrap_env_fn=wrapper.wrap_for_brax_training)
    print(f"{agent.upper()} {env_name} total time: {datetime.now() - t0}")

    return inference_fn, trained_params, metrics, rewards

# def visualize_rewards(_rewards, method_name = "PPO", env_name = "CartpoleBalance"):
#   _steps, _r = zip(*_rewards)

#   plt.plot(ppo_steps, ppo_r, label=method_name)
#   plt.plot(_steps,_r, label='')
#   plt.xlabel('Environment steps')
#   plt.ylabel('Episode reward')
#   plt.title(f'{env_name} - PPO vs SAC')
#   plt.legend()
#   plt.savefig('ppo_vs_sac.png')
#   plt.show()

## PPO

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('CartpoleBalance', 'ppo', 0)

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 1100.12 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 95a0b8b load on device 'cuda:0' took 7537.34 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint 981fdaf load on device 'cuda:0' took 4128.92 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.forward f60f76d load on device 'cuda:0' took 2504.77 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 1a5e55c load on device 'cuda:0' took 2279.06 ms  (compiled)
Module mujoco

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('CartpoleBalance', 'ppo', 26)

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 1024.96 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 95a0b8b load on device 'cuda:0' took 8414.09 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint 981fdaf load on device 'cuda:0' took 3167.10 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.forward f60f76d load on device 'cuda:0' took 2433.75 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 1a5e55c load on device 'cuda:0' took 2727.40 ms  (compiled)
Module mujoco

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('CartpoleBalance', 'ppo', 10)

[PPO|CartpoleBalance] steps=0, reward=263.35
[PPO|CartpoleBalance] steps=9830400, reward=442.01
[PPO|CartpoleBalance] steps=19660800, reward=994.73
[PPO|CartpoleBalance] steps=29491200, reward=502.77
[PPO|CartpoleBalance] steps=39321600, reward=988.56
[PPO|CartpoleBalance] steps=49152000, reward=999.52
[PPO|CartpoleBalance] steps=58982400, reward=999.64
[PPO|CartpoleBalance] steps=68812800, reward=999.66
[PPO|CartpoleBalance] steps=78643200, reward=999.72
[PPO|CartpoleBalance] steps=88473600, reward=999.66
PPO CartpoleBalance total time: 0:15:07.458773


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('ReacherEasy', 'ppo', 0)

[PPO|ReacherEasy] steps=0, reward=59.05
[PPO|ReacherEasy] steps=9830400, reward=957.41
[PPO|ReacherEasy] steps=19660800, reward=973.70
[PPO|ReacherEasy] steps=29491200, reward=975.49
[PPO|ReacherEasy] steps=39321600, reward=977.79
[PPO|ReacherEasy] steps=49152000, reward=979.26
[PPO|ReacherEasy] steps=58982400, reward=979.72
[PPO|ReacherEasy] steps=68812800, reward=981.07
[PPO|ReacherEasy] steps=78643200, reward=981.33
[PPO|ReacherEasy] steps=88473600, reward=981.10
PPO ReacherEasy total time: 0:18:59.291016


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('ReacherEasy', 'ppo', 26)

[PPO|ReacherEasy] steps=0, reward=67.45
[PPO|ReacherEasy] steps=9830400, reward=958.03
[PPO|ReacherEasy] steps=19660800, reward=972.23
[PPO|ReacherEasy] steps=29491200, reward=980.11
[PPO|ReacherEasy] steps=39321600, reward=982.22
[PPO|ReacherEasy] steps=49152000, reward=981.95
[PPO|ReacherEasy] steps=58982400, reward=982.29
[PPO|ReacherEasy] steps=68812800, reward=982.55
[PPO|ReacherEasy] steps=78643200, reward=983.34
[PPO|ReacherEasy] steps=88473600, reward=983.21
PPO ReacherEasy total time: 0:18:08.862944


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('ReacherEasy', 'ppo', 10)

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 1127.90 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 95a0b8b load on device 'cuda:0' took 7649.27 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint 981fdaf load on device 'cuda:0' took 4098.54 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.forward f60f76d load on device 'cuda:0' took 2445.74 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 1a5e55c load on device 'cuda:0' took 1954.98 ms  (compiled)
Module mujoco

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('SwimmerSwimmer6', 'ppo', 0)

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 578.09 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 95a0b8b load on device 'cuda:0' took 7893.19 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint 981fdaf load on device 'cuda:0' took 3675.84 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.sensor 7a62d05 load on device 'cuda:0' took 8551.03 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.forward f60f76d load on device 'cuda:0' took 2550.95 ms  (compiled)
Module mujoco.m

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('SwimmerSwimmer6', 'ppo', 26)

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 694.27 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 95a0b8b load on device 'cuda:0' took 7554.63 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint 981fdaf load on device 'cuda:0' took 4242.44 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.sensor 7a62d05 load on device 'cuda:0' took 7914.72 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.forward f60f76d load on device 'cuda:0' took 3371.03 ms  (compiled)
Module mujoco.m

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('SwimmerSwimmer6', 'ppo', 10)

[PPO|SwimmerSwimmer6] steps=0, reward=153.05
[PPO|SwimmerSwimmer6] steps=19660800, reward=355.69
[PPO|SwimmerSwimmer6] steps=39321600, reward=369.40
[PPO|SwimmerSwimmer6] steps=58982400, reward=368.76
[PPO|SwimmerSwimmer6] steps=78643200, reward=402.69
[PPO|SwimmerSwimmer6] steps=98304000, reward=402.97
[PPO|SwimmerSwimmer6] steps=117964800, reward=405.35
[PPO|SwimmerSwimmer6] steps=137625600, reward=443.20
[PPO|SwimmerSwimmer6] steps=157286400, reward=428.38
[PPO|SwimmerSwimmer6] steps=176947200, reward=464.82
PPO SwimmerSwimmer6 total time: 0:48:50.383849


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('HopperHop', 'ppo', 0)

Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 176.58 ms  (compiled)
Module _nxn_broadphase__locals__kernel_d2d200c6 d2d200c load on device 'cuda:0' took 631.05 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_ca3f8f98 ca3f8f9 load on device 'cuda:0' took 1161.59 ms  (compiled)
Module _efc_contact_init__locals__kernel_703a08a4 703a08a load on device 'cuda:0' took 224.33 ms  (compiled)
Module _efc_contact_jac_dense__locals__kernel_11a6f18c 927acba load on device 'cuda:0' took 1010.04 ms  (compiled)
Module _efc_contact_update__locals__kernel_c77ca81d c77ca81 load on device 'cuda:0' took 534.77 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.sensor 7a62d05 load on device 'cuda:0' took 8813.25 ms  (compiled)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_f956cf88 226c0f7 load on device 'cuda:0' took 5745.39 ms  (compiled)
Module solve_init_jaref__locals__kernel_173cbb76 1

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('HopperHop', 'ppo', 26)

Module _primitive_narrowphase__locals__primitive_narrowphase_ca3f8f98 ca3f8f9 load on device 'cuda:0' took 1088.46 ms  (compiled)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_f956cf88 226c0f7 load on device 'cuda:0' took 5548.78 ms  (compiled)
Module solve_init_jaref__locals__kernel_173cbb76 173cbb7 load on device 'cuda:0' took 183.10 ms  (compiled)
Module mul_m_dense__locals___mul_m_dense_d21c2e6d d21c2e6 load on device 'cuda:0' took 264.14 ms  (compiled)
Module update_constraint_gauss_cost__locals__kernel_b1449efe b1449ef load on device 'cuda:0' took 182.85 ms  (compiled)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_83d2c797 52c34bc load on device 'cuda:0' took 584.23 ms  (compiled)
Module update_gradient_cholesky__locals__kernel_6f68b037 18ece50 load on device 'cuda:0' took 4429.03 ms  (compiled)
Module linesearch_iterative__locals__kernel_c913459b 8ae51cc load on device 'cuda:0' took 1130.67 ms  (compiled)
[PPO|HopperHop] steps=0, reward=0.02


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('HopperHop', 'ppo', 10)

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 1222.84 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 95a0b8b load on device 'cuda:0' took 7355.22 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 148.40 ms  (compiled)
Module _nxn_broadphase__locals__kernel_d2d200c6 d2d200c load on device 'cuda:0' took 540.83 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_ca3f8f98 ca3f8f9 load on device 'cuda:0' took 1035.14 ms  (compiled)
M

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('WalkerWalk', 'ppo', 0)

Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_cdb981f2 c75d2e6 load on device 'cuda:0' took 4964.03 ms  (compiled)
Module solve_init_jaref__locals__kernel_051c9471 051c947 load on device 'cuda:0' took 154.99 ms  (compiled)
Module mul_m_dense__locals___mul_m_dense_86090c8a 86090c8 load on device 'cuda:0' took 294.63 ms  (compiled)
Module update_constraint_gauss_cost__locals__kernel_0aec6e48 0aec6e4 load on device 'cuda:0' took 205.75 ms  (compiled)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_2b23959a 84da6db load on device 'cuda:0' took 682.97 ms  (compiled)
Module update_gradient_cholesky__locals__kernel_97d1b30b 6b32db2 load on device 'cuda:0' took 5426.91 ms  (compiled)
Module linesearch_iterative__locals__kernel_6962ca73 89f17c9 load on device 'cuda:0' took 1182.78 ms  (compiled)
[PPO|WalkerWalk] steps=0, reward=32.85
[PPO|WalkerWalk] steps=9830400, reward=199.11
[PPO|WalkerWalk] steps=19660800, reward=723.72
[PPO|WalkerWalk] steps=29491200, re

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('WalkerWalk', 'ppo', 26)

Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 162.44 ms  (compiled)
Module _nxn_broadphase__locals__kernel_d2d200c6 d2d200c load on device 'cuda:0' took 617.22 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_304f8b71 304f8b7 load on device 'cuda:0' took 652.02 ms  (compiled)
Module _efc_contact_init__locals__kernel_703a08a4 703a08a load on device 'cuda:0' took 198.30 ms  (compiled)
Module _efc_contact_jac_dense__locals__kernel_11a6f18c 927acba load on device 'cuda:0' took 682.56 ms  (compiled)
Module _efc_contact_update__locals__kernel_c77ca81d c77ca81 load on device 'cuda:0' took 362.18 ms  (compiled)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_cdb981f2 c75d2e6 load on device 'cuda:0' took 5400.45 ms  (compiled)
Module solve_init_jaref__locals__kernel_051c9471 051c947 load on device 'cuda:0' took 236.23 ms  (compiled)
Module mul_m_dense__locals___mul_m_dense_86090c8a 86090c8 

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('WalkerWalk', 'ppo', 10)

[PPO|WalkerWalk] steps=0, reward=32.97
[PPO|WalkerWalk] steps=9830400, reward=206.87
[PPO|WalkerWalk] steps=19660800, reward=611.02
[PPO|WalkerWalk] steps=29491200, reward=779.72
[PPO|WalkerWalk] steps=39321600, reward=860.11
[PPO|WalkerWalk] steps=49152000, reward=958.11
[PPO|WalkerWalk] steps=58982400, reward=966.04
[PPO|WalkerWalk] steps=68812800, reward=965.80
[PPO|WalkerWalk] steps=78643200, reward=971.21
[PPO|WalkerWalk] steps=88473600, reward=970.75
PPO WalkerWalk total time: 0:28:56.096233


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('CheetahRun', 'ppo', 0)

Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 544.40 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 95a0b8b load on device 'cuda:0' took 9769.30 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 158.68 ms  (compiled)
Module _nxn_broadphase__locals__kernel_d2d200c6 d2d200c load on device 'cuda:0' took 538.21 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_ca3f8f98 ca3f8f9 load on device 'cuda:0' took 1014.36 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint 981fdaf load on device 'cuda:0' took 3390.11 ms  (compiled)
Module _efc_contact_init__locals__kernel_703a08a4 703a08a load on device 'cuda:0' took 280.75 ms  (compiled)
Module _efc_contact_jac_dense__locals__kernel_11a6f18c 927acba load on device 'cuda:0' took 1448.00 ms  (compiled)
Module _efc_contact_update__locals__kernel_c77ca81d c77ca81 load on device 'cuda:0' took 359.11 ms

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('CheetahRun', 'ppo', 26)

Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_cdb981f2 c75d2e6 load on device 'cuda:0' took 4890.72 ms  (compiled)
Module solve_init_jaref__locals__kernel_051c9471 051c947 load on device 'cuda:0' took 164.28 ms  (compiled)
Module mul_m_dense__locals___mul_m_dense_86090c8a 86090c8 load on device 'cuda:0' took 308.67 ms  (compiled)
Module update_constraint_gauss_cost__locals__kernel_0aec6e48 0aec6e4 load on device 'cuda:0' took 188.91 ms  (compiled)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_2b23959a 84da6db load on device 'cuda:0' took 611.85 ms  (compiled)
Module update_gradient_cholesky__locals__kernel_97d1b30b 6b32db2 load on device 'cuda:0' took 5414.29 ms  (compiled)
Module linesearch_iterative__locals__kernel_94acdc55 9c914fd load on device 'cuda:0' took 1126.92 ms  (compiled)
[PPO|CheetahRun] steps=0, reward=3.33
[PPO|CheetahRun] steps=9830400, reward=284.13
[PPO|CheetahRun] steps=19660800, reward=550.97
[PPO|CheetahRun] steps=29491200, rew

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('CheetahRun', 'ppo', 10)

[PPO|CheetahRun] steps=0, reward=2.90
[PPO|CheetahRun] steps=9830400, reward=310.23
[PPO|CheetahRun] steps=19660800, reward=626.78
[PPO|CheetahRun] steps=29491200, reward=768.31
[PPO|CheetahRun] steps=39321600, reward=860.33
[PPO|CheetahRun] steps=49152000, reward=552.09
[PPO|CheetahRun] steps=58982400, reward=136.33
[PPO|CheetahRun] steps=68812800, reward=2.74
[PPO|CheetahRun] steps=78643200, reward=2.06
[PPO|CheetahRun] steps=88473600, reward=19.56
PPO CheetahRun total time: 0:20:21.647289


## SAC

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('CartpoleBalance', 'sac', 0)

Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_2f3988cb 280e0ec load on device 'cuda:0' took 4204.20 ms  (compiled)
Module solve_init_jaref__locals__kernel_1024e45f 1024e45 load on device 'cuda:0' took 137.64 ms  (compiled)
Module mul_m_dense__locals___mul_m_dense_9e18c742 9e18c74 load on device 'cuda:0' took 211.68 ms  (compiled)
Module update_constraint_gauss_cost__locals__kernel_426b0947 426b094 load on device 'cuda:0' took 142.09 ms  (compiled)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_db6adfeb 2cf3ef2 load on device 'cuda:0' took 633.06 ms  (compiled)
Module update_gradient_cholesky__locals__kernel_2ebbd9c5 764264a load on device 'cuda:0' took 4785.84 ms  (compiled)
Module linesearch_iterative__locals__kernel_52ef4169 9b3d95e load on device 'cuda:0' took 1079.65 ms  (compiled)
[SAC|CartpoleBalance] steps=0, reward=199.03
[SAC|CartpoleBalance] steps=562944, reward=641.62
[SAC|CartpoleBalance] steps=1117696, reward=969.72
[SAC|CartpoleBalance]

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('CartpoleBalance', 'sac', 26)

Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_2f3988cb 280e0ec load on device 'cuda:0' took 4281.82 ms  (compiled)
Module solve_init_jaref__locals__kernel_1024e45f 1024e45 load on device 'cuda:0' took 139.20 ms  (compiled)
Module mul_m_dense__locals___mul_m_dense_9e18c742 9e18c74 load on device 'cuda:0' took 197.38 ms  (compiled)
Module update_constraint_gauss_cost__locals__kernel_426b0947 426b094 load on device 'cuda:0' took 140.34 ms  (compiled)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_db6adfeb 2cf3ef2 load on device 'cuda:0' took 539.08 ms  (compiled)
Module update_gradient_cholesky__locals__kernel_2ebbd9c5 764264a load on device 'cuda:0' took 4340.33 ms  (compiled)
Module linesearch_iterative__locals__kernel_52ef4169 9b3d95e load on device 'cuda:0' took 1691.91 ms  (compiled)
[SAC|CartpoleBalance] steps=0, reward=238.18
[SAC|CartpoleBalance] steps=562944, reward=940.44
[SAC|CartpoleBalance] steps=1117696, reward=896.57
[SAC|CartpoleBalance]

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('CartpoleBalance', 'sac', 10)

[SAC|CartpoleBalance] steps=0, reward=248.54
[SAC|CartpoleBalance] steps=562944, reward=548.79
[SAC|CartpoleBalance] steps=1117696, reward=975.49
[SAC|CartpoleBalance] steps=1672448, reward=904.48
[SAC|CartpoleBalance] steps=2227200, reward=985.72
[SAC|CartpoleBalance] steps=2781952, reward=991.64
[SAC|CartpoleBalance] steps=3336704, reward=991.15
[SAC|CartpoleBalance] steps=3891456, reward=987.87
[SAC|CartpoleBalance] steps=4446208, reward=991.32
[SAC|CartpoleBalance] steps=5000960, reward=994.05
SAC CartpoleBalance total time: 0:06:20.350038


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('ReacherEasy', 'sac', 0)

[SAC|ReacherEasy] steps=0, reward=59.99
[SAC|ReacherEasy] steps=562944, reward=974.01
[SAC|ReacherEasy] steps=1117696, reward=959.99
[SAC|ReacherEasy] steps=1672448, reward=964.77
[SAC|ReacherEasy] steps=2227200, reward=955.01
[SAC|ReacherEasy] steps=2781952, reward=960.83
[SAC|ReacherEasy] steps=3336704, reward=975.25
[SAC|ReacherEasy] steps=3891456, reward=982.52
[SAC|ReacherEasy] steps=4446208, reward=968.53
[SAC|ReacherEasy] steps=5000960, reward=947.97
SAC ReacherEasy total time: 0:07:16.345920


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('ReacherEasy', 'sac', 26)

[SAC|ReacherEasy] steps=0, reward=60.54
[SAC|ReacherEasy] steps=562944, reward=954.84
[SAC|ReacherEasy] steps=1117696, reward=950.62
[SAC|ReacherEasy] steps=1672448, reward=973.77
[SAC|ReacherEasy] steps=2227200, reward=951.78
[SAC|ReacherEasy] steps=2781952, reward=976.37
[SAC|ReacherEasy] steps=3336704, reward=978.88
[SAC|ReacherEasy] steps=3891456, reward=974.28
[SAC|ReacherEasy] steps=4446208, reward=952.07
[SAC|ReacherEasy] steps=5000960, reward=963.77
SAC ReacherEasy total time: 0:07:24.530146


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('ReacherEasy', 'sac', 10)

[SAC|ReacherEasy] steps=0, reward=44.17
[SAC|ReacherEasy] steps=562944, reward=956.80
[SAC|ReacherEasy] steps=1117696, reward=980.76
[SAC|ReacherEasy] steps=1672448, reward=975.24
[SAC|ReacherEasy] steps=2227200, reward=982.38
[SAC|ReacherEasy] steps=2781952, reward=958.88
[SAC|ReacherEasy] steps=3336704, reward=968.61
[SAC|ReacherEasy] steps=3891456, reward=952.73
[SAC|ReacherEasy] steps=4446208, reward=973.52
[SAC|ReacherEasy] steps=5000960, reward=956.34
SAC ReacherEasy total time: 0:07:22.665757


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('SwimmerSwimmer6', 'sac', 0)

[SAC|SwimmerSwimmer6] steps=0, reward=105.72
[SAC|SwimmerSwimmer6] steps=1118464, reward=184.32
[SAC|SwimmerSwimmer6] steps=2228736, reward=315.29
[SAC|SwimmerSwimmer6] steps=3339008, reward=260.50
[SAC|SwimmerSwimmer6] steps=4449280, reward=313.24
[SAC|SwimmerSwimmer6] steps=5559552, reward=259.60
[SAC|SwimmerSwimmer6] steps=6669824, reward=273.25
[SAC|SwimmerSwimmer6] steps=7780096, reward=213.78
[SAC|SwimmerSwimmer6] steps=8890368, reward=301.87
[SAC|SwimmerSwimmer6] steps=10000640, reward=257.69
SAC SwimmerSwimmer6 total time: 0:23:27.920612


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('SwimmerSwimmer6', 'sac', 26)

Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_f9126636 4908da2 load on device 'cuda:0' took 5523.68 ms  (compiled)
Module solve_init_jaref__locals__kernel_6e8468b3 6e8468b load on device 'cuda:0' took 154.10 ms  (compiled)
Module mul_m_dense__locals___mul_m_dense_f619f9d8 f619f9d load on device 'cuda:0' took 253.49 ms  (compiled)
Module update_constraint_gauss_cost__locals__kernel_84cf3b0a 84cf3b0 load on device 'cuda:0' took 179.11 ms  (compiled)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_dc766f91 aa19e9d load on device 'cuda:0' took 592.91 ms  (compiled)
Module update_gradient_cholesky__locals__kernel_8204ed89 a39c2f0 load on device 'cuda:0' took 4667.77 ms  (compiled)
[SAC|SwimmerSwimmer6] steps=0, reward=161.53
[SAC|SwimmerSwimmer6] steps=1118464, reward=197.77
[SAC|SwimmerSwimmer6] steps=2228736, reward=229.03
[SAC|SwimmerSwimmer6] steps=3339008, reward=199.23
[SAC|SwimmerSwimmer6] steps=4449280, reward=255.75
[SAC|SwimmerSwimmer6] steps=555

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('SwimmerSwimmer6', 'sac', 10)

[SAC|SwimmerSwimmer6] steps=0, reward=143.44
[SAC|SwimmerSwimmer6] steps=1118464, reward=167.37
[SAC|SwimmerSwimmer6] steps=2228736, reward=290.38
[SAC|SwimmerSwimmer6] steps=3339008, reward=250.04
[SAC|SwimmerSwimmer6] steps=4449280, reward=237.08
[SAC|SwimmerSwimmer6] steps=5559552, reward=188.18
[SAC|SwimmerSwimmer6] steps=6669824, reward=269.22
[SAC|SwimmerSwimmer6] steps=7780096, reward=282.33
[SAC|SwimmerSwimmer6] steps=8890368, reward=178.60
[SAC|SwimmerSwimmer6] steps=10000640, reward=195.00
SAC SwimmerSwimmer6 total time: 0:23:17.978614


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('HopperHop', 'sac', 0)

Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_f956cf88 226c0f7 load on device 'cuda:0' took 5189.17 ms  (compiled)
Module solve_init_jaref__locals__kernel_173cbb76 173cbb7 load on device 'cuda:0' took 153.97 ms  (compiled)
Module mul_m_dense__locals___mul_m_dense_d21c2e6d d21c2e6 load on device 'cuda:0' took 252.72 ms  (compiled)
Module update_constraint_gauss_cost__locals__kernel_b1449efe b1449ef load on device 'cuda:0' took 160.10 ms  (compiled)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_83d2c797 52c34bc load on device 'cuda:0' took 540.67 ms  (compiled)
Module update_gradient_cholesky__locals__kernel_6f68b037 18ece50 load on device 'cuda:0' took 4260.59 ms  (compiled)
Module linesearch_iterative__locals__kernel_c913459b 8ae51cc load on device 'cuda:0' took 1084.08 ms  (compiled)
[SAC|HopperHop] steps=0, reward=0.04
[SAC|HopperHop] steps=1118464, reward=1.75
[SAC|HopperHop] steps=2228736, reward=53.27
[SAC|HopperHop] steps=3339008, reward=95.46

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('HopperHop', 'sac', 26)

[SAC|HopperHop] steps=0, reward=0.01
[SAC|HopperHop] steps=1118464, reward=0.60
[SAC|HopperHop] steps=2228736, reward=54.63
[SAC|HopperHop] steps=3339008, reward=68.77
[SAC|HopperHop] steps=4449280, reward=72.87
[SAC|HopperHop] steps=5559552, reward=69.83
[SAC|HopperHop] steps=6669824, reward=79.69
[SAC|HopperHop] steps=7780096, reward=94.87
[SAC|HopperHop] steps=8890368, reward=116.30
[SAC|HopperHop] steps=10000640, reward=99.55
SAC HopperHop total time: 0:23:14.326198


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('HopperHop', 'sac', 10)

[SAC|HopperHop] steps=0, reward=0.13
[SAC|HopperHop] steps=1118464, reward=66.91
[SAC|HopperHop] steps=2228736, reward=115.40
[SAC|HopperHop] steps=3339008, reward=92.53
[SAC|HopperHop] steps=4449280, reward=117.24
[SAC|HopperHop] steps=5559552, reward=102.57
[SAC|HopperHop] steps=6669824, reward=116.96
[SAC|HopperHop] steps=7780096, reward=108.12
[SAC|HopperHop] steps=8890368, reward=122.27
[SAC|HopperHop] steps=10000640, reward=143.22
SAC HopperHop total time: 0:23:13.924325


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('WalkerWalk', 'sac', 0)

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 1336.90 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 95a0b8b load on device 'cuda:0' took 8746.99 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 164.93 ms  (compiled)
Module _nxn_broadphase__locals__kernel_d2d200c6 d2d200c load on device 'cuda:0' took 559.90 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_304f8b71 304f8b7 load on device 'cuda:0' took 614.66 ms  (compiled)
Mo

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('WalkerWalk', 'sac', 26)

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 1298.22 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 95a0b8b load on device 'cuda:0' took 8536.22 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 151.50 ms  (compiled)
Module _nxn_broadphase__locals__kernel_d2d200c6 d2d200c load on device 'cuda:0' took 577.41 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_304f8b71 304f8b7 load on device 'cuda:0' took 639.63 ms  (compiled)
Mo

In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('WalkerWalk', 'sac', 10)

[SAC|WalkerWalk] steps=0, reward=36.05
[SAC|WalkerWalk] steps=562944, reward=965.10
[SAC|WalkerWalk] steps=1117696, reward=972.77
[SAC|WalkerWalk] steps=1672448, reward=973.47
[SAC|WalkerWalk] steps=2227200, reward=973.07
[SAC|WalkerWalk] steps=2781952, reward=973.98
[SAC|WalkerWalk] steps=3336704, reward=972.27
[SAC|WalkerWalk] steps=3891456, reward=972.73
[SAC|WalkerWalk] steps=4446208, reward=975.18
[SAC|WalkerWalk] steps=5000960, reward=975.79
SAC WalkerWalk total time: 0:11:47.170364


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('CheetahRun', 'sac', 0)

Module _primitive_narrowphase__locals__primitive_narrowphase_ca3f8f98 ca3f8f9 load on device 'cuda:0' took 1057.93 ms  (compiled)
Module linesearch_iterative__locals__kernel_94acdc55 9c914fd load on device 'cuda:0' took 1127.97 ms  (compiled)
[SAC|CheetahRun] steps=0, reward=1.66
[SAC|CheetahRun] steps=1118464, reward=660.27
[SAC|CheetahRun] steps=2228736, reward=732.02
[SAC|CheetahRun] steps=3339008, reward=785.99
[SAC|CheetahRun] steps=4449280, reward=814.12
[SAC|CheetahRun] steps=5559552, reward=863.39
[SAC|CheetahRun] steps=6669824, reward=891.68
[SAC|CheetahRun] steps=7780096, reward=905.12
[SAC|CheetahRun] steps=8890368, reward=909.54
[SAC|CheetahRun] steps=10000640, reward=912.37
SAC CheetahRun total time: 0:21:42.125104


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('CheetahRun', 'sac', 26)

Module _primitive_narrowphase__locals__primitive_narrowphase_ca3f8f98 ca3f8f9 load on device 'cuda:0' took 1411.11 ms  (compiled)
Module linesearch_iterative__locals__kernel_94acdc55 9c914fd load on device 'cuda:0' took 1147.50 ms  (compiled)
[SAC|CheetahRun] steps=0, reward=28.31
[SAC|CheetahRun] steps=1118464, reward=673.47
[SAC|CheetahRun] steps=2228736, reward=744.28
[SAC|CheetahRun] steps=3339008, reward=790.62
[SAC|CheetahRun] steps=4449280, reward=869.09
[SAC|CheetahRun] steps=5559552, reward=891.72
[SAC|CheetahRun] steps=6669824, reward=905.64
[SAC|CheetahRun] steps=7780096, reward=912.64
[SAC|CheetahRun] steps=8890368, reward=915.67
[SAC|CheetahRun] steps=10000640, reward=917.71
SAC CheetahRun total time: 0:21:49.777024


In [ ]:
ppo_inference_fn, ppo_params, ppo_metrics, ppo_rewards = train_agent('CheetahRun', 'sac', 10)

[SAC|CheetahRun] steps=0, reward=2.88
[SAC|CheetahRun] steps=1118464, reward=673.72
[SAC|CheetahRun] steps=2228736, reward=746.08
[SAC|CheetahRun] steps=3339008, reward=812.84
[SAC|CheetahRun] steps=4449280, reward=859.65
[SAC|CheetahRun] steps=5559552, reward=889.52
[SAC|CheetahRun] steps=6669824, reward=903.33
[SAC|CheetahRun] steps=7780096, reward=910.40
[SAC|CheetahRun] steps=8890368, reward=915.69
[SAC|CheetahRun] steps=10000640, reward=908.75
SAC CheetahRun total time: 0:21:45.876615
